In [1]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC — loadings on CI Basis
# Expected: Voluntary Loading = 0.20, Standalone Loading = 0.20
# Handover Appendix A: "around row 141"
# ─────────────────────────────────────────────────────────────
from pathlib import Path
from openpyxl import load_workbook

BASE   = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE = BASE / "Basis and Assumptions.xlsx"

wb = load_workbook(SOURCE, data_only=True, read_only=False)
ws = wb["CI Basis"]

def find_anchors(ws, text):
    hits = []
    target = text.strip().lower()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip().lower() == target:
                hits.append((cell.row, cell.column))
    return hits

# ── 1) Probe candidate anchor labels ───────────────────────────
print("═══ Loading label probe (case-insensitive) ═══")
candidates = [
    "Voluntary Loading", "Voluntary loading", "Voluntary",
    "Standalone Loading", "Standalone loading", "Standalone",
    "Loading", "Loadings",
]
for label in candidates:
    hits = find_anchors(ws, label)
    if hits:
        print(f"   {label!r:>22} → {hits}")
    else:
        print(f"   {label!r:>22} → (no hits)")

# ── 2) Dump the area around row 141 as handover suggests ───────
print("\n═══ CI Basis rows 138–148, cols 1–8 ═══")
for r in range(138, 149):
    vals = []
    for c in range(1, 9):
        v = ws.cell(row=r, column=c).value
        if v is None:
            vals.append("·")
        elif isinstance(v, float):
            vals.append(f"{v:.4g}")
        else:
            vals.append(str(v)[:18])
    print(f"   r{r:>3}: " + " | ".join(f"{s:>18}" for s in vals))

# ── 3) Sweep for 0.2 values on the sheet (as an escape hatch) ──
print("\n═══ Cells containing exactly 0.2 on CI Basis (first 20) ═══")
found = []
for row in ws.iter_rows():
    for cell in row:
        v = cell.value
        if isinstance(v, (int, float)) and abs(v - 0.2) < 1e-9:
            found.append((cell.row, cell.column, v))
            if len(found) >= 20:
                break
    if len(found) >= 20:
        break
for r, c, v in found:
    label_left = ws.cell(row=r, column=c - 1).value if c > 1 else None
    label_above = ws.cell(row=r - 1, column=c).value if r > 1 else None
    print(f"   (r{r}, c{c}) = {v}   "
          f"left={label_left!r}   above={label_above!r}")

═══ Loading label probe (case-insensitive) ═══
      'Voluntary Loading' → (no hits)
      'Voluntary loading' → (no hits)
              'Voluntary' → (no hits)
     'Standalone Loading' → (no hits)
     'Standalone loading' → (no hits)
             'Standalone' → [(172, 15)]
                'Loading' → [(196, 24)]
               'Loadings' → (no hits)

═══ CI Basis rows 138–148, cols 1–8 ═══
   r138:                 84 |                  0 |                  0 |                  · |                 84 |                  0 |                  0 |                  ·
   r139:                 85 |                  0 |                  0 |                  · |                 85 |                  0 |                  0 |                  ·
   r140:                 86 |                  0 |                  0 |                  · |                 86 |                  0 |                  0 |                  ·
   r141:                 87 |                  0 |                  0 |        

In [2]:
# ─────────────────────────────────────────────────────────────
# VERIFY — dump rows 168–176, cols 14–20 to see full context
# around the two 0.2 values at (171, 17) and (172, 17)
# ─────────────────────────────────────────────────────────────
print("═══ CI Basis rows 168–176, cols 14–20 ═══")
for r in range(168, 177):
    vals = []
    for c in range(14, 21):
        v = ws.cell(row=r, column=c).value
        if v is None:
            vals.append("·")
        elif isinstance(v, float):
            vals.append(f"{v:.4g}")
        else:
            vals.append(str(v)[:20])
    print(f"   r{r:>3}: " + " | ".join(f"{s:>20}" for s in vals))

═══ CI Basis rows 168–176, cols 14–20 ═══
   r168:                    · |                    · |                    · |                    · |                    · |                    · |                    ·
   r169:                    · |                    · |                    · |                    · |                    · |                    · |                    ·
   r170:                    · |            Loadings: |                    · |                    · |                    · |                    · |                    ·
   r171:                    · |   Voluntary Loading: |                    · |                  0.2 |                    · |                    · |                    ·
   r172:                    · |           Standalone |                    · |                  0.2 |                    · |                    · |                    ·
   r173:                    · |                    · |                    · |                    · |                  